# Paggawa ng Mga Aplikasyon para sa Pagbuo ng Larawan (OpenAI)

Hindi lang tekstong pagbuo ang kaya ng LLMs. Maaari ka ring bumuo ng mga larawan mula sa mga paglalarawan sa teksto. Ang mga larawan bilang isang modality ay kapaki-pakinabang sa MedTech, arkitektura, turismo, pag-develop ng laro, marketing, at iba pa. Sa lekisyong ito ay gagamit tayo ng mga **GPT Image** na modelo sa OpenAI platform.

## Mga Layunin sa Pag-aaral

Sa dulo ng leksiyong ito ay magagawa mo na:

- Ipaliwanag kung ano ang pagbuo ng larawan at saan ito kapaki-pakinabang.
- Maunawaan ang pamilya ng modelo na `gpt-image` at kung paano ito naiiba sa mga legacy na modelo ng DALL·E.
- Gumawa ng aplikasyon sa pagbuo ng larawan at mag-edit ng mga larawan.

## Ano ang pagbuo ng larawan?

Ang mga modelo sa pagbuo ng larawan ay lumilikha ng mga larawan mula sa isang text prompt. Ang mga modernong modelo gaya ng `gpt-image` ay natututo ng ugnayan sa pagitan ng teksto at mga larawan habang nagsasanay, at paulit-ulit nitong ginagawang imahe mula sa random na ingay na tumutugma sa iyong prompt.

Dalawang kilalang pamilya ng mga modelo ng larawan ay:

- **`gpt-image` (OpenAI)** — ang kasalukuyang henerasyon na ginagamit sa leksiyong ito. Sinusuportahan nito ang text-to-image generation at pag-edit ng larawan (inpainting gamit ang mask).
- **Midjourney** — isang sikat na third-party model na may sariling serbisyo at workflow sa Discord.

> Ang mga luma nang OpenAI image models — **DALL·E 2** at **DALL·E 3** — ay mga legacy. Ang DALL·E 3 ay hindi na magagamit para sa mga bagong deployment, at ang mga tampok tulad ng `create_variation` ay lamang umiiral sa DALL·E 2. Gamitin ang mga modelo ng `gpt-image` para sa mga bagong aplikasyon.

> **Mahalaga:** Ang mga modelo ng `gpt-image` ay nagbabalik ng binuong larawan bilang **base64** (`b64_json`), hindi bilang URL. Kinakailangang i-decode ng iyong code ang base64 string sa mga byte at i-save ito — walang URL ng larawan na mada-download.


## Paggawa ng iyong unang application para sa pagbuo ng larawan

Ano nga ba ang kailangan para makagawa ng application para sa pagbuo ng larawan? Kailangan mo ang mga sumusunod na library:

- **python-dotenv**, lubos na inirerekomenda na gamitin ang library na ito para ilagay ang iyong mga lihim sa isang *.env* na file na hiwalay sa code.
- **openai**, ito ang library na gagamitin mo para makipag-interact sa OpenAI API.
- **pillow**, para sa pagtrabaho gamit ang mga larawan sa Python.


1. Gumawa ng isang file na *.env* na may sumusunod na nilalaman:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Ipunin ang mga nabanggit na library sa isang file na tinatawag na *requirements.txt* tulad nito:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Susunod, gumawa ng virtual environment at i-install ang mga library:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Para sa Windows, gamitin ang mga sumusunod na utos para gumawa at i-activate ang iyong virtual na kapaligiran:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Idagdag ang sumusunod na code sa isang file na tinatawag na *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Gumawa ng OpenAI object (binabasa ang OPENAI_API_KEY mula sa iyong .env)
    client = openai.OpenAI()


    try:
        # Gumawa ng larawan gamit ang image generation API
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Ilagay dito ang iyong prompt text
            size='1024x1024',
            n=1
        )
        # Itakda ang direktoryo para sa nakaimbak na larawan
        image_dir = os.path.join(os.curdir, 'images')

        # Kung ang direktoryo ay wala pa, gawin ito
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # I-initialize ang path ng larawan (tandaan na ang filetype ay dapat png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Ang mga gpt-image model ay nagbabalik ng larawan bilang base64 (b64_json), hindi URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Ipakita ang larawan sa default na image viewer
        image = Image.open(image_path)
        image.show()

    # hulihin ang mga exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Ipaliwanag natin ang code na ito:

- Una, ini-import natin ang mga library na kailangan natin, kabilang ang OpenAI library, dotenv library, base64 module, at Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Pagkatapos noon, gumawa tayo ng client, na nagbabasa ng API key mula sa iyong ``.env``.

    ```python
    # Gumawa ng OpenAI na bagay
    client = openai.OpenAI()
    ```

- Sunod, ginagawa natin ang imahe:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Ipasok ang iyong teksto ng prompt dito
        size='1024x1024',
        n=1
    )
    ```

    Ang `gpt-image` models ay nagbabalik ng imahe bilang isang **base64** na string sa `data[0].b64_json`. Ini-decode natin ito sa bytes at sinusulat sa isang file — walang URL para i-download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Sa huli, binubuksan natin ang imahe at ginagamit ang karaniwang image viewer para ipakita ito:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Mga karagdagang detalye sa paggawa ng imahe

Tingnan natin ang mga parameter ng `images.generate`:

- **model**, ay ang image model na gagamitin (halimbawa `gpt-image-1`).
- **prompt**, ay ang text prompt na ginagamit para gumawa ng imahe. Dito ay “Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils”.
- **size**, ay ang sukat ng ginawang imahe (halimbawa `1024x1024`, `1536x1024`, `1024x1536`, o `"auto"`).
- **n**, ay ang bilang ng mga imaheng ginawa. Dito ay gumagawa tayo ng isa.

> Hindi tumatanggap ang mga image model ng `temperature` parameter — ito ay kontrol para sa pagbuo ng teksto. Para makakuha ng iba't ibang resulta, tawagin muli ang API; para bumaba ang pagkakaiba, gawing mas tiyak ang iyong prompt.

## Karagdagang kakayahan sa paggawa ng imahe

Nakita mo kung paano gumawa ng imahe gamit ang ilang linya ng Python. Ang mga `gpt-image` models ay kayang **i-edit** ang isang umiiral na imahe. Sa pamamagitan ng pagbibigay ng imahe, isang opsyonal na **mask** (na nagmamarka sa bahagi na babaguhin), at isang prompt, maaari mong baguhin ang bahagi ng imahe — halimbawa, magdagdag ng sumbrero sa ating kuneho.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# ang mga pagbabago ay ibinabalik din bilang base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Ang base na imahe ay naglalaman lamang ng kuneho; ang panghuling imahe ay may sumbrero sa kuneho.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
